# HerdNet Minimal working example

In [ ]:
!pip install \
    timm==1.0.22 \
    albumentations==1.0.3 \
    hydra-core==1.3.2 \
    opencv-python==4.10.0.84 \
    pillow==10.4.0 \
    scikit-image \
    scikit-learn \
    wandb==0.20.1 \
    gdown==5.2.0 \
    matplotlib==3.10.8 \
    tqdm==4.67.1 \
    loguru==0.7.3 \
    seaborn==0.13.2 \
    numpy==2.4.1 \
    scipy==1.16.0 \
    pandas==2.3.1

In [ ]:
## Installation

### Installation Colab

In [ ]:
# Download and install the code
import sys

!git clone https://github.com/cwinkelmann/HerdNet
!cd '/content/HerdNet' && python setup.py install

sys.path.append('/content/HerdNet')

### Install using the conda environment

In [4]:
!conda env create -n HerdNet -f ../environment.yml

Channels:
 - pytorch
 - defaults
 - conda-forge
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.0
    latest version: 25.11.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



scipy-1.16.0         | 22.7 MB   |                                       |   0% 
pandas-2.3.1         | 15.0 MB   |                                       |   0% 

numpy-base-2.4.1     | 6.9 MB    |                                       |   0% 


libopenblas-0.3.30   | 4.1 MB    |                                       |   0% 



libgfortran5-15.2.0  | 742 KB    |                                       |   0% 




llvm-openmp-20.1.8   | 327 KB    |                                       |   0% 





numexpr-2.14.1       | 193 KB    |                                       |   0% 






bottleneck-1.4.2     | 122 KB    |                                       |   0% 







numpy-2.4.1          | 20 KB     |    


CondaError: Run 'conda init' before 'conda activate'



In [ ]:
## Optional update if the environment changed
!conda env update --file environment.yml --prune


## (Optional) Install Active Learning Repository
This contains functions for Training Data Preparation, Geospatial Inference and Detection Deduplication.
TODO

In [ ]:
import sys
sys.path.append('./')

In [4]:
from pathlib import Path
Path("./").resolve()

PosixPath('/home/cwinkelmann/work/HerdNet/notebooks')

# TODO

In [4]:
from loguru import logger

logger.disable("animaloc")

### Train a model using checkppints

In [ ]:
from pathlib import Path
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf

In [ ]:
## Create a config by hand:

# Create config
cfg = OmegaConf.create({
    "losses": {
        "FocalLoss": {
            "print_name": "focal_loss",
            "from_torch": False,
            "output_idx": 0,
            "target_idx": 0,
            "lambda_const": 1.0,
            "kwargs": {
                "alpha": 2,
                "beta": 4,
                "reduction": "mean",
                "normalize": False,
            }
        },
        "CrossEntropyLoss": {
            "print_name": "ce_loss",
            "from_torch": True,
            "output_idx": 1,
            "target_idx": 1,
            "lambda_const": 1.0,
            "background_class_weight": 0.1,
            "kwargs": {
                "reduction": "mean",
                "weight": [0.1, 5, 0.1],
            }
        }
    },
    "datasets": {
        "img_size": [512, 512],
        "anno_type": "point",
        "num_classes": 3,
        "collate_fn": None,
        "class_def": {
            1: "iguana_point",
            2: "hard_negative",
        },
        "train": {
            "name": "CSVDataset",
            "csv_file": None,
            "root_dir": None,
            "sampler": None,
        },
        "validate": {
            "name": "CSVDataset",
            "csv_file": None,
            "root_dir": None,
            "albu_transforms": {"Normalize": {"p": 1.0}},
            "end_transforms": {"DownSample": {"down_ratio": 4, "anno_type": "point"}},
        },
        "test": {
            "name": "CSVDataset",
            "csv_file": None,
            "root_dir": str(DATA_DIR),
            "albu_transforms": {"Normalize": {"p": 1.0}},
            "end_transforms": {"DownSample": {"down_ratio": 4, "anno_type": "point"}},
        },
    },
    "training_settings": {
        "trainer": "Trainer",
        "batch_size": 12,
        "num_workers": 2,
        "evaluator": {
            "name": "HerdNetEvaluator",
            "threshold": 100,
            "select_mode": "max",
            "validate_on": "f1_score",
            "kwargs": {
                "print_freq": 125,
                "lmds_kwargs": {
                    "kernel_size": [9, 9],
                    "adapt_ts": 0.3,
                    "scale_factor": 1,
                    "up": True,
                }
            }
        },
        "stitcher": {
            "name": "HerdNetStitcher",
            "kwargs": {
                "overlap": 120,
                "down_ratio": 4,
                "up": False,
                "reduction": "mean",
            }
        },
    },
    "model": {
        "name": "CamouflageHerdNetConvNeXt",
        "from_torchvision": False,
        "load_from": str(MODEL_PATH),
        "resume_from": None,
        "kwargs": {
            "pretrained": True,
            "down_ratio": 4,
            "backbone_size": "base",
        },
        "freeze": None,
    },
    "wandb_flag": False,
    "seed": 1,
    "device_name": None,  # auto-detect
})

In [1]:


from animaloc.utils.train import main

# Clear any previous Hydra state (important in notebooks when re-running cells)
GlobalHydra.instance().clear()

# Configure paths
config_dir = str(Path.cwd() / "configs" / "demo")  # adjust as needed
config_name = "dla34_delplanque"

# Load config
with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name=config_name, overrides=[
        # Add any overrides here, e.g.:
        # "training.epochs=10",
        # "training.batch_size=4",
    ])

# Inspect config (optional)
print(OmegaConf.to_yaml(cfg))



/home/cwinkelmann/miniconda3/envs/HerdNetCarrotConda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


losses:
  FocalLoss:
    print_name: focal_loss
    from_torch: false
    output_idx: 0
    target_idx: 0
    lambda_const: 1.0
    kwargs:
      alpha: 2
      beta: 4
      reduction: mean
      normalize: false
  CrossEntropyLoss:
    print_name: ce_loss
    from_torch: true
    output_idx: 1
    target_idx: 1
    lambda_const: 1.0
    background_class_weight: 0.1
    kwargs:
      reduction: mean
      weight:
      - ${losses.CrossEntropyLoss.background_class_weight}
      - 0.24
      - 0.15
      - 0.248
      - 0.045
      - 0.02
      - 0.28
datasets:
  img_size:
  - 512
  - 512
  anno_type: point
  num_classes: 7
  collate_fn: null
  class_def:
    1: Alcelaphinae
    2: Buffalo
    3: Kob
    4: Warthog
    5: Waterbuck
    6: Elephant
  train:
    name: CSVDataset
    csv_file: /raid/cwinkelmann/training_data/iguana/2025_08_10_endgame/Floreana_detection_il_9/train/herdnet_format_512_0_crops.csv
    root_dir: /raid/cwinkelmann/training_data/iguana/2025_08_10_endgame/Floreana

In [2]:
# Run training
result = main(cfg)
wandb.finish()

2026-01-28 17:18:08.467 | INFO     | animaloc.utils.train:main:45 - Using config: {'losses': {'FocalLoss': {'print_name': 'focal_loss', 'from_torch': False, 'output_idx': 0, 'target_idx': 0, 'lambda_const': 1.0, 'kwargs': {'alpha': 2, 'beta': 4, 'reduction': 'mean', 'normalize': False}}, 'CrossEntropyLoss': {'print_name': 'ce_loss', 'from_torch': True, 'output_idx': 1, 'target_idx': 1, 'lambda_const': 1.0, 'background_class_weight': 0.1, 'kwargs': {'reduction': 'mean', 'weight': ['${losses.CrossEntropyLoss.background_class_weight}', 0.24, 0.15, 0.248, 0.045, 0.02, 0.28]}}}, 'datasets': {'img_size': [512, 512], 'anno_type': 'point', 'num_classes': 7, 'collate_fn': None, 'class_def': {1: 'Alcelaphinae', 2: 'Buffalo', 3: 'Kob', 4: 'Warthog', 5: 'Waterbuck', 6: 'Elephant'}, 'train': {'name': 'CSVDataset', 'csv_file': '/raid/cwinkelmann/training_data/iguana/2025_08_10_endgame/Floreana_detection_il_9/train/herdnet_format_512_0_crops.csv', 'root_dir': '/raid/cwinkelmann/training_data/iguana/2

[TRAINING] - Epoch: [1] [ 1/12] eta: 0:00:11 lr: 0.000009 loss: 30.7774 (30.7774) focal_loss: 28.6014 (28.6014) ce_loss: 2.1760 (2.1760) time: 0.9666 data: 0.1702 max mem: 0


2026-01-28 17:18:18.587 | INFO     | animaloc.train.trainers:_train:833 - [TRAINING] - Epoch: [1] mean loss: 15.7040


[TRAINING] - Epoch: [1] [12/12] eta: 0:00:00 lr: 0.000100 loss: 10.3152 (15.7040) focal_loss: 9.8316 (14.8387) ce_loss: 0.4947 (0.8653) time: 0.1292 data: 0.0174 max mem: 0
[TRAINING] - Epoch: [1] Total time: 0:00:01 (0.1315 s / it)
[VALIDATION] - Epoch: [1] [ 1/56] eta: 0:01:47 n: 12 tp: 5 fp: 3 fn: 4 recall: 0.56 precision: 0.62 f1_score: 0.59 f2_score: 0.57 f5_score: 0.56 MAE: 1.0 ME: -3.0 MSE: 1.0 RMSE: 1.0 avg_score: 0.72 avg_dscore: 0.163 time: 1.9125 data: 0.0095 max mem: 0


/home/cwinkelmann/work/Herdnet/animaloc/eval/metrics.py:338: RuntimeWarning: invalid value encountered in divide
  recalls = sorted_table[:,2] / n_gt
2026-01-28 17:18:24.310 | INFO     | animaloc.train.trainers:start:327 - [VALIDATION] - Epoch: [1] f1_score: 0.3964
2026-01-28 17:18:24.311 | INFO     | animaloc.train.trainers:start:436 - Checking for best model by Evaluator output at epoch 1 with validation output: 0.3964365256124721


...

2026-01-28 17:18:46.518 | INFO     | animaloc.train.trainers:_train:833 - [TRAINING] - Epoch: [5] mean loss: 2.7548


[TRAINING] - Epoch: [5] [12/12] eta: 0:00:00 lr: 0.000100 loss: 2.2803 (6.6245) focal_loss: 2.2540 (6.3303) ce_loss: 0.0263 (0.2942) time: 0.0620 data: 0.0187 max mem: 0
[TRAINING] - Epoch: [5] Total time: 0:00:00 (0.0670 s / it)
[VALIDATION] - Epoch: [5] [ 1/56] eta: 0:01:37 n: 9 tp: 9 fp: 0 fn: 0 recall: 1.0 precision: 1.0 f1_score: 1.0 f2_score: 1.0 f5_score: 1.0 MAE: 0.0 ME: 244.0 MSE: 0.0 RMSE: 0.0 avg_score: 0.95 avg_dscore: 0.362 time: 1.7452 data: 0.0111 max mem: 0


2026-01-28 17:18:52.384 | INFO     | animaloc.train.trainers:start:327 - [VALIDATION] - Epoch: [5] f1_score: 0.9953
2026-01-28 17:18:52.385 | INFO     | animaloc.train.trainers:start:436 - Checking for best model by Evaluator output at epoch 5 with validation output: 0.9952606635071091


[VALIDATION] - Epoch: [5] [56/56] eta: 0:00:00 n: 1 tp: 1 fp: 0 fn: 0 recall: 1.0 precision: 1.0 f1_score: 1.0 f2_score: 1.0 f5_score: 1.0 MAE: 0.0 ME: 243.0 MSE: 0.0 RMSE: 0.0 avg_score: 0.96 avg_dscore: 0.367 time: 0.1379 data: 0.0065 max mem: 0
[VALIDATION] - Epoch: [5] Total time: 0:00:05 (0.1047 s / it)


2026-01-28 17:18:52.765 | INFO     | animaloc.train.trainers:start:441 - Best model by End User Metric f1_score saved - Epoch 5 - Validation value: 0.995261, path: /home/cwinkelmann/work/Herdnet/best_model.pth
2026-01-28 17:18:52.766 | INFO     | animaloc.train.trainers:start:476 - Checking for best model by validation loss at epoch 5 with validation output: None
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
/home/cwinkelmann/work/Herdnet/animaloc/utils/train.py:405: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details

### Inference with the model




In [5]:
# Inference notebook cell

# Inference notebook cell

from pathlib import Path
from hydra import initialize_config_dir, compose
from hydra.core.global_hydra import GlobalHydra

from animaloc.utils.inference import inference

# Clear Hydra state
GlobalHydra.instance().clear()

config_dir = str(Path.cwd() / "configs" / "demo")  # adjust as needed
config_name = "dla34_delplanque"


# Load the just trained model and load a folder with images
with initialize_config_dir(config_dir=config_dir, version_base="1.1"):
    cfg = compose(config_name=config_name, overrides=[
        "model.load_from=/home/cwinkelmann/work/Herdnet/best_model.pth",
        "datasets.test.root_dir=/home/cwinkelmann/work/Herdnet/tests/data/single_images/ISWF01_22012023_subset",
        # add other overrides as needed
    ])

# Run inference
detections = inference(cfg, plain_inference=True, vis_detections=False)

# Show results
print(f"Total detections: {len(detections)}")
detections.head(10)

wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
/home/cwinkelmann/work/Herdnet/animaloc/models/utils.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded

[TEST] [1/1] eta: 0:00:19 n: 23 tp: 0 fp: 22 fn: 1 recall: 0.0 precision: 0.0 f1_score: 0.0 f2_score: 0.0 f5_score: 0.0 MAE: 21.0 ME: 21.0 MSE: 441.0 RMSE: 21.0 avg_score: 0.36 avg_dscore: 0.138 time: 19.8834 data: 0.4550 max mem: 216
[TEST] Total time: 0:00:19 (19.8855 s / it)
Wandb summary: <wandb.sdk.wandb_summary.Summary object at 0x7f7f6a364650>


wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


Total detections: 22


,images,labels,scores,dscores,x,y,count_1,count_2,count_3,count_4,count_5,count_6,species
0,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.067305,0.140227,3474.0,204.0,22,0,0,0,0,0,Alcelaphinae
1,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.148564,0.114275,2780.0,328.0,22,0,0,0,0,0,Alcelaphinae
2,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.015158,0.106460,3628.0,574.0,22,0,0,0,0,0,Alcelaphinae
3,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.280455,0.108733,4878.0,712.0,22,0,0,0,0,0,Alcelaphinae
4,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.000010,0.124402,3090.0,744.0,22,0,0,0,0,0,Alcelaphinae
5,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.002116,0.141932,3166.0,848.0,22,0,0,0,0,0,Alcelaphinae
6,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.918238,0.191062,4782.0,858.0,22,0,0,0,0,0,Alcelaphinae
7,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.588382,0.129851,3858.0,1136.0,22,0,0,0,0,0,Alcelaphinae
8,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.950966,0.174142,4424.0,1136.0,22,0,0,0,0,0,Alcelaphinae
9,Isa_ISWF01_DJI_0057_22012023.JPG,1,0.160941,0.144296,2690.0,1250.0,22,0,0,0,0,0,Alcelaphinae


## Find optimal class weights to cope with class imbalance

In [19]:
import yaml

class_counts = df['labels'].value_counts()

# 2. Convert counts to frequencies (i.e., fraction of the total)
class_freqs = class_counts / class_counts.sum()

# 3. Compute inverse frequency for each class
class_weights_inv = 1.0 / class_freqs

# 4. Convert to a dictionary {class_id: weight_value}
class_weights_dict = class_weights_inv.to_dict()

print("Class distribution:\n", class_freqs)
print("\nInverse frequency weights:\n", class_weights_dict)

class_weights_dict = class_weights_inv.to_dict()

# 5. Save to a YAML file
with open('class_weights.yaml', 'w') as f:
    yaml.safe_dump(class_weights_dict, f, sort_keys=True)

Class distribution:
 labels
3     0.803690
2     0.085936
4     0.035772
6     0.029912
5     0.014811
7     0.011720
8     0.010400
1     0.003252
13    0.001288
15    0.000934
11    0.000869
14    0.000708
10    0.000515
9     0.000161
12    0.000032
Name: count, dtype: float64

Inverse frequency weights:
 {3: 1.2442610472336846, 2: 11.636568002997377, 4: 27.954995499549955, 6: 33.431646932185146, 5: 67.51739130434783, 7: 85.32417582417582, 8: 96.15479876160991, 1: 307.5049504950495, 13: 776.4499999999999, 15: 1070.9655172413793, 11: 1150.2962962962963, 14: 1411.7272727272727, 10: 1941.1249999999998, 9: 6211.599999999999, 12: 31057.999999999996}
